In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import spacy
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim.downloader as api

# Loading spaCy English model
nlp = spacy.load('en_core_web_sm')

# Download VADER lexicon (only once)
nltk.download('vader_lexicon')

# Load a pretraiend Word2Vec model (Google news, 300d)
print("Loading pretrained Word2Vec model (this might take a few minutes)")
w2v_model = api.load("word2vec-google-news-300")

print("Libraries imported successfully!")

Libraries imported successfully!


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Barsam\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [7]:
# Loading the transcribed data (first 10 rows only for development)

file_path = r"C:\Uni\transcribed_data_assemblyAI.xlsx"
df = pd.read_excel(file_path)

# Keep only first 10 rows for development and testing
df = df.head(10)

# Basic info about the dataset
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("\nFirst 5 rows:")
df.head()

Dataset shape: (10, 1)
Columns: ['Sentence']

First 5 rows:


,Sentence
0,Previously on Australian Survivor
1,Tara and Locky have been close allies since da...
2,I'm trusting you with this because I'm sitting...
3,But Tara decided it was time for a breakup
4,"It breaks my heart, but he's got to go"


In [8]:
# Get the sentences as a list for easier processing

sentences = df['Sentence'].tolist()

# Display some basic statistics
print(f"Total sentences: {len(sentences)}")
print(f"Sample sentences:")
for i in range(5):
    print(f"{i+1}: {sentences[i]}")

Total sentences: 10
Sample sentences:
1: Previously on Australian Survivor
2: Tara and Locky have been close allies since day one
3: I'm trusting you with this because I'm sitting on the outs on this one
4: But Tara decided it was time for a breakup
5: It breaks my heart, but he's got to go


# 1: Part-of-Speech (POS) Tagging

In [9]:
# Simpler POS tagging approach
def get_pos_tags(sentence):
    """Get POS tags for one sentence"""
    doc = nlp(sentence)
    return [token.pos_ for token in doc]

# # #  Apply to all sentences 

df['POS_Tags'] = df['Sentence'].apply(get_pos_tags)

# # # Printing all results 

print("POS tagging complete!")
print(f"Example: {df['Sentence'].iloc[0]}")
print(f"POS Tags: {df['POS_Tags'].iloc[0]}")

POS tagging complete!
Example: Previously on Australian Survivor
POS Tags: ['ADV', 'ADP', 'PROPN', 'PROPN']


# 2: Term Frequency-Inverse Document Frequency (TF-IDF):

In [10]:
def extract_tfidf(sentences):
    """Find which words are important in each sentence"""
    
    # Make a tool to find important words
    # Keep only top 1000 words
    # Skip boring words like "the" and "and"
    vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
    
    # Look at all sentences and find important words
    tfidf_matrix = vectorizer.fit_transform(sentences)
    
    # Change the results to simple numbers
    tfidf_dense = tfidf_matrix.toarray()
    tfidf_list = [row.tolist() for row in tfidf_dense]
    
    return tfidf_list, vectorizer

# Find important words in our sentences
print("Finding important words...")
sentences_list = df['Sentence'].tolist()
tfidf_features, tfidf_vectorizer = extract_tfidf(sentences_list)

# Put results in our table
df['TF_IDF'] = tfidf_features

# Show what we got
print(f"Each sentence has {len(tfidf_features[0])} number scores")
print(f"First sentence scores: {tfidf_features[0][:5]}")

# Show which words are important
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"Important words we found: {list(feature_names[:10])}")

Finding important words...
Each sentence has 34 number scores
First sentence scores: [0.0, 0.5773502691896257, 0.0, 0.0, 0.0]
Important words we found: ['allies', 'australian', 'bad', 'blindside', 'breaks', 'breakup', 'close', 'day', 'decided', 'dilemma']


# Part 3: Sentiment Analysis

In [26]:
#initialize analyzer
sia = SentimentIntensityAnalyzer()

#apply to each sentence
df["Sentiment_Score"] = df["Sentence"].apply(lambda x: sia.polarity_scores(str(x))["compound"])

# Show result
print(df[["Sentence", "Sentiment_Score"]].head(10))


                                            Sentence  Sentiment_Score
0                  Previously on Australian Survivor           0.3612
1  Tara and Locky have been close allies since da...           0.0000
2  I'm trusting you with this because I'm sitting...           0.4019
3         But Tara decided it was time for a breakup           0.0000
4             It breaks my heart, but he's got to go           0.0000
5        It wasn't the only moral dilemma Tara faced          -0.1779
6                 My eyes just popped out of my head           0.0000
7                              I don't even feel bad           0.4310
8  Tara sweet talked the tribe into voting with h...           0.4588
9                            I want to blindside him           0.0772


# Part 4: Pretrained Word Embeddings

In [30]:
# Sentence to vector function using spaCy tokenizer to accomodate for other languages
def sentence_to_vectore(sentence, model, nlp): 
    doc = nlp(str(sentence)) # tokenize sentences using spaCy
    vecs = [model[token.text] for token in doc if token.text in model]
    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis = 0)

# Apply to dataset
df["Word2Vec_Vector"] = df["Sentence"].apply(lambda x: sentence_to_vectore(x, w2v_model, nlp))

# Show the results
for i, row in df.iterrows():
    print(f"\nRow{i}")
    print("Sentence:", row["Sentence"])
    print("Vector shape:", row["Word2Vec_Vector"][:10])


Row0
Sentence: Previously on Australian Survivor
Vector shape: [-0.02017212 -0.20013428 -0.03469849  0.0690918  -0.01333618  0.00109863
  0.05783081 -0.08337402  0.17651367  0.03045654]

Row1
Sentence: Tara and Locky have been close allies since day one
Vector shape: [ 2.03314889e-02 -6.50634766e-02 -8.19227472e-03  1.22633189e-01
  1.35633682e-05 -4.70513254e-02 -9.04032364e-02 -4.15852852e-02
  1.81783035e-01  1.15234375e-01]

Row2
Sentence: I'm trusting you with this because I'm sitting on the outs on this one
Vector shape: [ 0.10599518  0.00395584  0.0567627   0.17524719 -0.02470398  0.00822449
  0.0369854  -0.11209106  0.07735825  0.08548164]

Row3
Sentence: But Tara decided it was time for a breakup
Vector shape: [-0.00310516  0.07085586 -0.01702118  0.04579926 -0.04064941 -0.05474854
  0.05101395 -0.14781189  0.13484955 -0.00169373]

Row4
Sentence: It breaks my heart, but he's got to go
Vector shape: [-0.003479    0.1298523   0.01425171  0.11933899 -0.07309723 -0.01370239
  0.0

### Explanation for what happens to words that are not represented in the pre-trained word embedding model

The words not represented in the pre trained model are found in the sentence to vector function, especially this part of the funciton 
<pre>
        if token.text in model
</pre>
<p>
this code checks whether or not the word exists in the pre trained library, if its not present in the library the word is skipped.
<br>
then later using the code
<pre>
        if len(vecs) == 0:
        return np.zeros(model.vector_size)
</pre>
we check if the all words in the sentence or missing, if that is the case the function returns a zero vector (so the shape stays consistent, but it means "no info available").

# Part 5: Custom Word Embeddings and Additional NLP Feature

In [1]:
import pandas as pd
import numpy as np
from collections import Counter

# 1) Load the CSV file (combined dataset for embeddings)
path = r"C:\Users\custo\Desktop\task 5 retake\combined_dataset.csv"

# Robust read: try auto-detect delimiter first, then fallback to semicolon
try:
    train_df = pd.read_csv(path, sep=None, engine="python")
except Exception:
    train_df = pd.read_csv(path, sep=";", engine="python")

# Clean column names (handles headers like ';sentence;emotion;')
train_df.columns = [str(c).strip().strip(";") for c in train_df.columns]

# If everything ended up in one column, split on ';'
if len(train_df.columns) == 1 and "sentence" in train_df.columns[0].lower():
    train_df = pd.read_csv(path, sep=";", engine="python")
    train_df.columns = [str(c).strip().strip(";") for c in train_df.columns]

# Drop empty/unnamed columns (caused by leading/trailing delimiters)
train_df = train_df.loc[:, ~train_df.columns.astype(str).str.match(r"^Unnamed")]
train_df = train_df.drop(columns=[c for c in train_df.columns if str(c).strip() == ""], errors="ignore")

print("Available columns:", train_df.columns.tolist())

# Use the correct text column
if "sentence" not in train_df.columns:
    raise KeyError(f"'sentence' column not found. Available columns: {train_df.columns.tolist()}")

texts = train_df["sentence"].dropna().astype(str)

# Fix encoding artifact seen in the preview ("Â")
texts = texts.str.replace("Â", "", regex=False)

print(f"\nLoaded {len(texts)} sentences")

# Data description
word_lists = [s.split() for s in texts]
all_words = [w for sent in word_lists for w in sent]

vocab = set(all_words)
avg_len = np.mean([len(s) for s in word_lists]) if word_lists else 0
max_len = np.max([len(s) for s in word_lists]) if word_lists else 0
min_len = np.min([len(s) for s in word_lists]) if word_lists else 0

print("\nDataset characteristics:")
print(f"Total sentences: {len(texts):,}")
print(f"Total words (tokens): {len(all_words):,}")
print(f"Vocabulary size: {len(vocab):,}")
print(f"Average sentence length: {avg_len:.2f} words")
print(f"Max sentence length: {max_len} words")
print(f"Min sentence length: {min_len} words")

print("\nTop 10 most frequent words:")
for word, count in Counter(all_words).most_common(10):
    print(f"  {word}: {count} times")

Available columns: ['sentence', 'emotion']

Loaded 53392 sentences

Dataset characteristics:
Total sentences: 53,392
Total words (tokens): 635,607
Vocabulary size: 61,449
Average sentence length: 11.90 words
Max sentence length: 69 words
Min sentence length: 1 words

Top 10 most frequent words:
  the: 17994 times
  I: 17871 times
  to: 13825 times
  a: 13409 times
  you: 9190 times
  and: 9083 times
  is: 8440 times
  of: 7568 times
  that: 7029 times
  in: 6024 times


In [2]:
from gensim.utils import simple_preprocess

# 2) Tokenize the training text (combined dataset)
sentences = [simple_preprocess(s, deacc=True, min_len=1) for s in texts]

# Safe sample print (avoids IndexError if the dataset is very small after cleaning)
sample_idx = 1 if len(sentences) > 1 else 0
print(f"Tokenized sample (index {sample_idx}):", sentences[sample_idx])

Tokenized sample (index 1): ['you', 'must', 've', 'had', 'your', 'hands', 'full']


### 3) Train Word2Vec on the combined training corpus (NOT the test episode)

**Rationale for hyperparameters (brief):**
- **`vector_size=100`**: A common baseline that balances representational capacity with the risk of overfitting on a medium-sized corpus.
- **`window=5`**: Captures local conversational context by looking at nearby words within a moderate context window.
- **`min_count=2`**: Filters out single-occurrence tokens to reduce noise and improve embedding quality on a larger dataset.
- **`sg=1` (skip-gram)**: Often learns better representations for less frequent words compared to CBOW.
- **`epochs=40`**: Uses more training passes to improve convergence and stabilize similarity results on a modest corpus.

In [ ]:
from gensim.models import Word2Vec
model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    epochs=40,
    workers=4  # optional: speeds up training
)

print("Vocabulary size:", len(model.wv))

Vocabulary size: 13720


In [4]:
probe_words = ["happy", "sad", "mad"]
for w in probe_words:
    if w in model.wv:
        print(f"\nMost similar to '{w}':")
        for sim, score in model.wv.most_similar(w, topn=5):
            print(f"  {sim:>15} : {score:.3f}")
    else:
        print(f"\nWord '{w}' not in vocabulary")


Most similar to 'happy':
          pleased : 0.537
     appreciative : 0.536
          uncanny : 0.523
         thankful : 0.520
              bop : 0.509

Most similar to 'sad':
       triggering : 0.569
           cringe : 0.545
        relatable : 0.540
         palpable : 0.533
     intimidating : 0.526

Most similar to 'mad':
     unattractive : 0.535
           aaaand : 0.529
     irrationally : 0.505
         destined : 0.504
            niche : 0.502


In [5]:
# 4.Make sentence embeddings
def sentence_vector(text):
    tokens = simple_preprocess(text)
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

# test on first sentence
vec = sentence_vector(texts.iloc[0])
print("Sentence embedding shape:", vec.shape)

Sentence embedding shape: (100,)


In [6]:
from numpy import dot
from numpy.linalg import norm

def cosine_similarity(a, b):
    return dot(a, b) / (norm(a) * norm(b))

s1 = "I am very happy today!"
s2 = "I feel so excited and joyful!"
s3 = "This is a boring challenge."

v1, v2, v3 = sentence_vector(s1), sentence_vector(s2), sentence_vector(s3)

print("happy vs excited:", cosine_similarity(v1, v2))  # should be high
print("happy vs boring :", cosine_similarity(v1, v3))  # should be low


happy vs excited: 0.57387334
happy vs boring : 0.4316094


# Part 6: Additional Feature: Emotion Lexicon (word & phrase counts)

In [7]:
import re
from gensim.utils import simple_preprocess

# Tiny, illustrative lexicons (extendable)
ANGER_WORDS   = {"furious","angry","mad","rage","raging","annoyed","irritated","pissed","upset"}
HAPPY_WORDS     = {"happy","happier","joy","joyful","excited","glad","thrilled","celebrate","smile","smiling"}
SAD_WORDS     = {"sad","sadder","saddest","cry","crying","tears","lonely","alone","heartbroken"}
FEAR_WORDS    = {"afraid","scared","fear","nervous","anxious","worry","worried","worries","worrying"}

# phrases common in the data (map each to an emotion bucket)
ANGER_PHRASES = ["mess with me", "go home", "get rid of"]
SAD_PHRASES   = ["left out"]

def _normalize_for_phrase(text: str) -> str:
    # lowercase, strip punctuation to spaces, collapse spaces
    t = re.sub(r"[^a-z0-9\s]+", " ", text.lower())
    t = re.sub(r"\s+", " ", t).strip()
    return t

def emotion_lexicon_feature(text: str) -> dict:
    raw = str(text)

    # token-level counts (uses gensim's simple_preprocess to drop punctuation)
    tokens = simple_preprocess(raw, deacc=True, min_len=1)
    counts = {
        "Anger_Words":   sum(1 for w in tokens if w in ANGER_WORDS),
        "Happy_Words":     sum(1 for w in tokens if w in HAPPY_WORDS),
        "Sadness_Words": sum(1 for w in tokens if w in SAD_WORDS),
        "Fear_Words":    sum(1 for w in tokens if w in FEAR_WORDS),
    }

    # phrase-level counts (matches across spaces, ignores punctuation/case)
    norm = _normalize_for_phrase(raw)
    counts["Anger_Phrases"]   = sum(1 for p in ANGER_PHRASES if f" {p} " in f" {norm} ")
    counts["Sadness_Phrases"] = sum(1 for p in SAD_PHRASES   if f" {p} " in f" {norm} ")

    # Total for convenience
    counts["EmotionLex_Total"] = (
        counts["Anger_Words"] + counts["Happy_Words"] + counts["Sadness_Words"] + counts["Fear_Words"]
        + counts["Anger_Phrases"] + counts["Sadness_Phrases"]
    )
    return counts


In [9]:
import pandas as pd
import numpy as np
from gensim.utils import simple_preprocess

def sentence_vector(text: str, model, vector_size: int) -> list:
    toks = simple_preprocess(str(text), deacc=True, min_len=1)
    vecs = [model.wv[w] for w in toks if w in model.wv]
    if not vecs:
        return np.zeros(vector_size, dtype=float).tolist()
    return np.mean(vecs, axis=0).tolist()

df = pd.read_csv("NLP_features.csv")
vec_size = model.vector_size

df["Word2Vec_Vector"] = (
    df["Sentence"]
      .astype(str)
      .apply(lambda s: sentence_vector(s, model, vec_size))
      .astype(str)
)

df.to_csv("NLP_features.csv", index=False)
print("Updated Word2Vec_Vector for ALL sentences using the new custom model.")

Updated Word2Vec_Vector for ALL sentences using the new custom model.
